In [1]:
import pandas as pd
import numpy as np
import os
import cv2
import sys
import torch
from torch.utils.data import Dataset, DataLoader
custom_module_path = os.path.join("/mnt/Main Drive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm")
sys.path.append(custom_module_path)

In [2]:
train_data = pd.read_csv('archive_2/Train.csv')
train_data.head(10)


,video_id,label,frames,label_id,shape,format
0,1,Doing other things,37,0,"(100, 176)",JPEG
1,3,Pushing Two Fingers Away,37,6,"(100, 176)",JPEG
2,6,Drumming Fingers,37,1,"(100, 176)",JPEG
3,11,Sliding Two Fingers Down,37,10,"(100, 176)",JPEG
4,14,Pushing Hand Away,37,5,"(100, 176)",JPEG
5,17,Shaking Hand,37,9,"(100, 176)",JPEG
6,20,Doing other things,37,0,"(100, 176)",JPEG
7,28,Pulling Two Fingers In,37,4,"(100, 176)",JPEG
8,31,Stop Sign,37,14,"(100, 176)",JPEG
9,34,Zooming In With Two Fingers,37,24,"(100, 176)",JPEG


In [3]:
train_data[['label', 'label_id']].value_counts().sort_index()

label                          label_id
Doing other things             0           4374
Drumming Fingers               1           1818
No gesture                     2           1844
Pulling Hand In                3           1829
Pulling Two Fingers In         4           1859
Pushing Hand Away              5           1812
Pushing Two Fingers Away       6           1843
Rolling Hand Backward          7           1715
Rolling Hand Forward           8           1788
Shaking Hand                   9           1789
Sliding Two Fingers Down       10          1832
Sliding Two Fingers Left       11          1816
Sliding Two Fingers Right      12          1780
Sliding Two Fingers Up         13          1779
Stop Sign                      14          1821
Swiping Down                   15          1824
Swiping Left                   16          1762
Swiping Right                  17          1730
Swiping Up                     18          1768
Thumb Down                     19          1810


In [5]:
!pip install pytorchvideo

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━ 23.1/35.4 MB 3.5 MB/s eta 0:00:04^C
   ━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━ 23.1/35.4 MB 3.5 MB/s eta 0:00:04
ERROR: Operation cancelled by user


In [4]:
import pytorch_lightning
import pytorchvideo.data
import torch.utils.data
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageOps
from enum import Enum
import pandas as pd

from torch import randint
from pathlib import Path


/home/neutrino/miniconda3/envs/Ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'pytorchvideo'

In [ ]:
class JesterDataset(Dataset):
    class FrameSelectStrategy(Enum):
        FROM_BEGINNING = 0
        FROM_END = 1
        RANDOM = 2

    class FramePadding(Enum):
        REPEAT_END = 0
        REPEAT_BEGINNING = 2

    def __init__(self, csv_file, video_dir, number_of_frames, frame_file_ending="jpg",
                 frame_select_strategy=FrameSelectStrategy.RANDOM, frame_padding=FramePadding.REPEAT_END,
                 video_transform=None):
        """
        A pytorch dataset to load the 20BN-JESTER dataset or datasets in the same format.

        Args:
            csv_file: Path to the csv file, describing the videos. In jester this is eg. "1;Swipe Right" where 1 is
                the video folder name
            video_dir: Path to the directory containing the videos
            frame_file_ending: File ending of the video images, eg. "jpg" for file names like "0001.jpg"
            number_of_frames: The number of frames to get from one video.
            frame_select_strategy: When the video has more frames than number_of_frames, then the frame_select_strategy
                is used to select a subset of the frames.
            frame_padding: If the video does not have enough frames, the frame padding strategy is used to fill the frames.
            video_transform:  Videotransforms to apply to the frames. You can not use the usual torchvision.transforms
                as the same transform must be applied to all the frames of one video.
                Checkout https://github.com/hassony2/torch_videovision for video compatible transforms.

        Example:
            from jesterdataset import JesterDataset
            from torch_videovision.videotransforms.volume_transforms import ClipToTensor

            dataset = JesterDataset("./jester_data/jester-v1-train.csv", "./jester_data/20bn-jester-v1",
                                    video_transform=ClipToTensor())
            dataloader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=4)

            for i_batch, sample_batched in enumerate(dataloader):
                self.assertLessEqual(len(sample_batched), 4)
        """
        self.file_ending = frame_file_ending
        self.video_dir = video_dir
        self.number_of_frames = number_of_frames
        self.frame_select_strategy = frame_select_strategy
        self.frame_padding = frame_padding

        self.video_transform = video_transform
        self.data_description = self._read_csv(csv_file)

    def _read_csv(self, path):
        data = pd.read_csv(path)
        return data

    def __getitem__(self, index):
        # print(f"\nAccessing index: {index}")
        # print(f"\nData description row: {self.data_description.iloc[index]}")
        
        video_id, label = self.data_description.iloc[index]['video_id'], self.data_description.iloc[index]['label_id']
        video_directory = Path(self.video_dir) / str(video_id)
        frame_files = list(Path(video_directory).glob(f"*.{self.file_ending}"))
        if len(frame_files) == 0:
            raise FileNotFoundError(f"Could not find any frames. There should be at least one frame in the directory "
                                    f"{video_directory}")
        frame_files = self._add_padding(frame_files, self.number_of_frames, self.frame_padding)
        frame_files = self._select_frames(frame_files, self.frame_select_strategy, self.number_of_frames)
        frames = [Image.open(frame_file).convert('RGB') for frame_file in frame_files]
        

        if self.video_transform:
            frames = [self.video_transform(frame) for frame in frames]
        
        frames_tensor = torch.stack([transforms.ToTensor()(frame) for frame in frames])

        return frames_tensor, label

    def __len__(self):
        return len(self.data_description)

    def _add_padding(self, frame_files, number_of_frames, frame_padding: FramePadding):
        difference = number_of_frames - len(frame_files)
        if difference > 0:
            if frame_padding == self.FramePadding.REPEAT_BEGINNING:
                frame_index_to_repeat = 0
            elif frame_padding == self.FramePadding.REPEAT_END:
                frame_index_to_repeat = -1
            else:
                raise ValueError("Frame Padding Type not supported")

            frame_files += [frame_files[frame_index_to_repeat] for _ in range(difference)]

        return frame_files

    def _select_frames(self, frame_files: list, frame_select_strategy: FrameSelectStrategy, number_of_frames: int):
        if len(frame_files) <= number_of_frames:
            return frame_files
        else:
            if frame_select_strategy == self.FrameSelectStrategy.FROM_BEGINNING:
                return frame_files[:number_of_frames]
            elif frame_select_strategy == self.FrameSelectStrategy.FROM_END:
                return frame_files[-number_of_frames:]
            elif frame_select_strategy == self.FrameSelectStrategy.RANDOM:
                difference = len(frame_files) - number_of_frames
                random_start_index = randint(0, difference, (1,)).item()
                end_index = random_start_index + number_of_frames
                return frame_files[random_start_index:end_index]
            else:
                raise ValueError("FrameSelectStrategy not supported.")

In [ ]:
from torch.utils.data.dataloader import DataLoader
from torchvision import transforms
from torchvideotransforms.volume_transforms import ClipToTensor

video_transform = transforms.Compose([
    # transforms.ToPILImage(),
    transforms.Resize((124, 124)),  # Resize frames to 224x224
    # transforms.ToTensor()
])

dataset_1 = JesterDataset(csv_file = "/mnt/Main Drive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/archive/Train_aug.csv", 
video_dir = "/mnt/Main Drive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/augmented/Train",number_of_frames = 32,
video_transform=video_transform)
train_loader = DataLoader(dataset_1, batch_size=32, shuffle=True, num_workers=8)

In [ ]:
dataset_2 = JesterDataset(csv_file = "/mnt/Main Drive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/archive/Validation_copy.csv", 
video_dir = "/mnt/Main Drive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/archive/Val",number_of_frames = 32,
video_transform=video_transform)
val_loader = DataLoader(dataset_2, batch_size=32, shuffle=True, num_workers=8)

In [ ]:
# Iterate through the dataloader
for i_batch, sample_batched in enumerate(train_loader):
    batch, label = sample_batched
    print(f"\nBatch number {i_batch} has a batch size of {len(batch)}")
    print(f"Batch type: {type(batch)}")
    
    # Print the shape of the batch tensor
    print(f"Batch shape: {batch.shape}")
    
    # Assuming batch is of shape (batch_size, num_frames, channels, height, width)
    batch_size, num_frames, channels, height, width = batch.shape
    for i in range(batch_size):
        print(f"Shape of video {i}: {batch[i].shape}")
    
    print(f"Labels: {label}")

In [ ]:
# Iterate through the dataloader
for i_batch, sample_batched in enumerate(val_loader):
    batch, label = sample_batched
    print(f"\nBatch number {i_batch} has a batch size of {len(batch)}")
    print(f"Batch type: {type(batch)}")
    
    # Print the shape of the batch tensor
    print(f"Batch shape: {batch.shape}")
    
    # Assuming batch is of shape (batch_size, num_frames, channels, height, width)
    batch_size, num_frames, channels, height, width = batch.shape
    for i in range(batch_size):
        print(f"Shape of video {i}: {batch[i].shape}")
    
    print(f"Labels: {label}")

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class EnhancedInceptionLSTMModel(nn.Module):
    def __init__(self, num_classes, seq_length, img_size):
        """
        Build an enhanced model with CNN and MaxPool layers before InceptionV3,
        followed by LSTM for sequence processing.

        Args:
            num_classes: Number of output classes
            seq_length: Number of frames per video
            img_size: Size of each frame (height, width)
        """
        super(EnhancedInceptionLSTMModel, self).__init__()
        
        self.seq_length = seq_length
        self.img_size = img_size
        self.min_size = 75
        
        # Upsampling layers if input size is smaller than the minimum size
        self.upsample = None
        if img_size[0] < self.min_size or img_size[1] < self.min_size:
            upsample_factor_h = max(1, self.min_size // img_size[0])
            upsample_factor_w = max(1, self.min_size // img_size[1])
            upsample_factor = max(upsample_factor_h, upsample_factor_w)
            self.upsample = nn.Upsample(scale_factor=upsample_factor, mode='bilinear', align_corners=False)
        
        # Custom CNN layers for preprocessing
        self.cnn_preprocess = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU()
        )
        
        # Load InceptionV3 model
        self.inception = models.inception_v3(pretrained=True, aux_logits=False)
        self.inception.fc = nn.Identity()  # Remove the fully connected layer
        for param in self.inception.parameters():
            param.requires_grad = False  # Freeze InceptionV3 weights
        
        # LSTM layers for sequence processing
        self.lstm = nn.LSTM(
            input_size=2048,  # Output size of InceptionV3
            hidden_size=512,
            num_layers=2,
            batch_first=True,
            dropout=0.2
        )
        
        # Fully connected layers for classification
        self.fc = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        """
        Forward pass of the model.

        Args:
            x: Input tensor of shape (batch_size, seq_length, channels, height, width)

        Returns:
            Output tensor of shape (batch_size, num_classes)
        """
        batch_size, seq_length, channels, height, width = x.size()
        
        # Process each frame in the sequence
        processed_frames = []
        for t in range(seq_length):
            frame = x[:, t]  # Extract the t-th frame
            
            # Upsample if needed
            if self.upsample:
                frame = self.upsample(frame)
            
            # Apply custom CNN preprocessing
            frame = self.cnn_preprocess(frame)
            
            # Extract features using InceptionV3
            frame_features = self.inception(frame)
            processed_frames.append(frame_features)
        
        # Stack processed frames into a sequence
        processed_frames = torch.stack(processed_frames, dim=1)  # Shape: (batch_size, seq_length, 2048)
        
        # Pass the sequence through LSTM
        lstm_out, _ = self.lstm(processed_frames)  # Shape: (batch_size, seq_length, 512)
        lstm_out = lstm_out[:, -1, :]  # Take the output of the last LSTM cell
        
        # Pass through fully connected layers
        output = self.fc(lstm_out)  # Shape: (batch_size, num_classes)
        
        return output

# Example usage
num_classes = 10
seq_length = 16
img_size = (64, 64)

model = EnhancedInceptionLSTMModel(num_classes=num_classes, seq_length=seq_length, img_size=img_size)
print(model)

In [ ]:
for name, param in model.named_parameters():
    print(name, param.shape)            

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import os
import matplotlib.pyplot as plt

os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

# Assuming model is already defined
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = torch.optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss()
num_epochs = 50  # Total number of epochs
checkpoint_dir = '/mnt/Main Drive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

def save_checkpoint(epoch, model, optimizer, loss, accuracy, checkpoint_dir):
    checkpoint_path = os.path.join(checkpoint_dir, f'checkpoint_epoch_{epoch+1}.pth')
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
        'accuracy': accuracy
    }, checkpoint_path)
    print(f'Checkpoint saved at {checkpoint_path}')

def load_checkpoint(checkpoint_path, model, optimizer):
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    epoch = checkpoint['epoch']
    loss = checkpoint['loss']
    accuracy = checkpoint['accuracy']
    print(f'Checkpoint loaded from {checkpoint_path}')
    return epoch, loss, accuracy

# Load checkpoint from epoch 10
start_epoch = 0
checkpoint_path = os.path.join(checkpoint_dir, f'checkpoint_epoch_{start_epoch}.pth')
if os.path.exists(checkpoint_path):
    start_epoch, _, _ = load_checkpoint(checkpoint_path, model, optimizer)
else:
    start_epoch = 0

train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []

for epoch in range(start_epoch, num_epochs):  # Start from the loaded epoch
    model.train()
    running_loss = 0.0
    correct_predictions = 0
    total_predictions = 0
    start_time = time.time()
    
    for i, (inputs, labels) in enumerate(train_loader):
        # Check the shape of the input tensor
        # print(f"Original input shape: {inputs.shape}")
        
        # Permute the dimensions if necessary
        if inputs.shape[1] == 32:  # Assuming 32 is the number of channels
            inputs = inputs.permute(0, 2, 1, 3, 4)  # Change shape to [batch_size, channels, num_frames, height, width]
        
        inputs = inputs.to(device)  # Move inputs to GPU
        labels = labels.to(device)  # Move labels to GPU
        # print("labels: ",labels)
        # print("Inputs: ",inputs)

        optimizer.zero_grad()
        outputs = model(inputs)
   
        # Debugging prints
        # print(f"Outputs shape: {outputs.shape}")
        # print(f"Labels shape: {labels.shape}")

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        
        # Calculate accuracy
        predicted = torch.argmax(outputs, 1)
        correct_predictions += (predicted == labels).sum().item()
        total_predictions += labels.size(0)
        
        if i % 1000 == 9:  # Print every 1000 batches
            print(f'Epoch [{epoch+1}/{num_epochs}], Batch [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}')
    
    epoch_time = time.time() - start_time
    epoch_accuracy = correct_predictions / total_predictions
    train_losses.append(running_loss / len(train_loader))
    train_accuracies.append(epoch_accuracy)

    print("=======================================================")
    print(f'Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}, Accuracy: {epoch_accuracy:.4f}, Time: {epoch_time:.2f}s')
    print("=======================================================")

    # Save checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        save_checkpoint(epoch, model, optimizer, running_loss/len(train_loader), epoch_accuracy, checkpoint_dir)

    # Validation
    model.eval()
    val_loss = 0.0
    correct_predictions = 0
    total_predictions = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            # Check the shape of the input tensor
            # print(f"Original input shape: {inputs.shape}")
            
            # Permute the dimensions if necessary
            if inputs.shape[1] == 32:  # Assuming 32 is the number of channels
                inputs = inputs.permute(0, 2, 1, 3, 4)  # Change shape to [batch_size, channels, num_frames, height, width]
            
            inputs = inputs.to(device)  # Move inputs to GPU
            labels = labels.to(device)  # Move labels to GPU if already tensor
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            # Calculate accuracy
            predicted = torch.argmax(outputs, 1)
            correct_predictions += (predicted == labels).sum().item()
            total_predictions += labels.size(0)
    
    val_accuracy = correct_predictions / total_predictions
    val_losses.append(val_loss / len(val_loader))
    val_accuracies.append(val_accuracy)

    print("=============================================================")
    print(f'Validation Loss: {val_loss/len(val_loader):.4f}, Validation Accuracy: {val_accuracy:.4f}')
    print("=============================================================")